In [1]:
!pip install --upgrade bitsandbytes transformers accelerate peft


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 6.1/12.0 MB 75.4 MB/s eta 0:00:01
ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone may have tampered with them.
    unknown package:
        Expected sha256 c77d353a4851b1880191603d36acb313411d3577f6e2897814f333841f7003f4
             Got        cb6d338e1b4c709c575ead293fa9090573d24fa05fbedd5844f19f66a0bf412c



In [ ]:
pip install transformers datasets accelerate peft torch pandas openpyxl sentencepiece evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=bfbbb217984212cef49d01243894e75231d11183355854f79726ff7846bb55e7
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [2]:
# Step 1: Import libraries
import pandas as pd
from google.colab import files

# Step 2: Upload your Excel file
uploaded = files.upload()  # This will open a file picker

# Step 3: Get the uploaded file name
file_name = list(uploaded.keys())[0]
print(f"Uploaded file: {file_name}")

# Step 4: Load the Excel file into a DataFrame
df = pd.read_excel(file_name)

# Step 5: Check the first few rows
print(df.head())
print(f"Dataset size: {len(df)}")


Saving A.xlsx to A.xlsx
Uploaded file: A.xlsx
      Type                                   User Requirement  \
0  Website  We need a simple website for our local bakery ...   
1      App  We want a basic habit tracker app for users to...   
2     Game  We need a simple 2D endless runner game for mo...   
3  Website  We need a medium-sized online forum website fo...   
4      App  We want a medium-complexity recipe app for hom...   

                                        User Stories  \
0  - As a customer, I want to browse cake flavors...   
1  - As a user, I want to add habits so I can tra...   
2  - As a player, I want to control the character...   
3  - As a user, I want to post photos so I can sh...   
4  - As a user, I want to search recipes so I can...   

                                    Module Breakdown  
0   - Login/Signup - Home Page - Menu Catalog - I...  
1   - User Login - Habit Addition UI - Reminders ...  
2   - Login/Signup - Game Engine - Character Cont...  
3   - 

In [42]:
import pandas as pd

df = pd.read_excel("A.xlsx")
print(df.head())
print(df.columns)
print(df.shape)
print(df.isna().sum())  # Check missing values

      Type                                   User Requirement  \
0  Website  We need a simple website for our local bakery ...   
1      App  We want a basic habit tracker app for users to...   
2     Game  We need a simple 2D endless runner game for mo...   
3  Website  We need a medium-sized online forum website fo...   
4      App  We want a medium-complexity recipe app for hom...   

                                        User Stories  \
0  - As a customer, I want to browse cake flavors...   
1  - As a user, I want to add habits so I can tra...   
2  - As a player, I want to control the character...   
3  - As a user, I want to post photos so I can sh...   
4  - As a user, I want to search recipes so I can...   

                                    Module Breakdown  
0   - Login/Signup - Home Page - Menu Catalog - I...  
1   - User Login - Habit Addition UI - Reminders ...  
2   - Login/Signup - Game Engine - Character Cont...  
3   - Login/Signup - Post & Upload Section - Comm...

In [43]:
# Drop rows where any of the 3 columns is missing
df = df.dropna(subset=['User Requirement', 'User Stories', 'Module Breakdown']).reset_index(drop=True)

# Clean text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = " ".join(text.split())  # remove extra spaces, newlines
    return text

df['User Requirement'] = df['User Requirement'].apply(clean_text)
df['User Stories'] = df['User Stories'].apply(clean_text)
df['Module Breakdown'] = df['Module Breakdown'].apply(clean_text)

# Remove empty rows after cleaning
df = df[(df['User Requirement'] != "") & (df['User Stories'] != "") & (df['Module Breakdown'] != "")].reset_index(drop=True)

print(f"Final dataset size: {len(df)}")

Final dataset size: 1958


In [44]:
instruction = "Convert the following user requirement into detailed User Stories and a Module Breakdown."

def create_prompt(row):
    input_text = row['User Requirement']

    target = f"""User Stories:
{row['User Stories']}

Module Breakdown:
{row['Module Breakdown']}"""

    return {
        "input_text": f"{instruction}\n\nUser Requirement: {input_text}",
        "output_text": target.strip()
    }

# Apply to dataset
examples = df.apply(create_prompt, axis=1).tolist()

# Convert to final format
final_data = [
    {"inputs": ex["input_text"], "targets": ex["output_text"]}
    for ex in examples
]

In [45]:
from datasets import Dataset

# Create dataset
data_dict = {
    "inputs": [ex["inputs"] for ex in final_data],
    "targets": [ex["targets"] for ex in final_data]
}

dataset = Dataset.from_dict(data_dict)
dataset = dataset.train_test_split(test_size=0.1)  # 90% train, 10% validation

In [46]:
from transformers import AutoTokenizer

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_input_length = 512
max_target_length = 512

def preprocess_function(examples):
    inputs = [ex for ex in examples["inputs"]]
    targets = [ex for ex in examples["targets"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding=False
    )

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_target_length,
            truncation=True,
            padding=False
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/1762 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/196 [00:00<?, ? examples/s]

In [10]:
!pip install --upgrade bitsandbytes transformers accelerate peft


  Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
Using cached bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl (59.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2


In [47]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Use 4-bit or LoRA if you have low GPU RAM (recommended)
# Option A: Full fine-tune (if you have 24GB+ GPU)
# Option B: LoRA (recommended for most people)

# === RECOMMENDED: Use LoRA (very efficient) ===
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig
import torch

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Should show ~1-3M trainable params

trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


In [48]:
# CRITICAL: Permanently disable torch.compile for the entire session
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"        # This kills Dynamo completely
os.environ["TORCH_COMPILE_DISABLE"] = "1"      # Extra safety

# Also disable any accidental torch.compile in Transformers
import torch
torch._dynamo.config.suppress_errors = True   # Prevents crashes, just falls back

In [49]:
# Add this AT THE TOP of your notebook (before any imports)
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"    # Disables TorchDynamo globally
os.environ["TORCH_COMPILE_DISABLE"] = "1"  # Disables torch.compile globally

In [50]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-userstories-finetuned",
    eval_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=12,
    predict_with_generate=True,
    fp16=False,                    # Required for 4-bit QLoRA
    bf16=False,                    # T4 doesn't support bf16
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # FIXED: Use "eval_loss" for seq2seq
    greater_is_better=False,
    report_to="none",
    generation_max_length=512,
    torch_compile=False,           # Still good to include (harmless)
    dataloader_num_workers=0,      # Prevents DataLoader crashes on Colab
    remove_unused_columns=False,   # Keeps dataset columns intact
)

In [51]:
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()  # Success! No more errors.

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,2.081000,1.691869
2,1.573400,1.345585
3,1.456700,1.218486
4,1.424300,1.135605
5,1.385500,1.084898
6,1.258800,1.045717
7,1.273600,1.022599
8,1.212900,1.002330
9,1.236600,0.987893
10,1.265800,0.979959


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=672, training_loss=1.4087232017800921, metrics={'train_runtime': 1702.1288, 'train_samples_per_second': 12.422, 'train_steps_per_second': 0.395, 'total_flos': 1656256011872256.0, 'train_loss': 1.4087232017800921, 'epoch': 12.0})

In [52]:
# FINAL INFERENCE – WORKS 100% WITH YOUR SAVED MODEL
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel, PeftConfig
from transformers import BitsAndBytesConfig
import torch
import os

# 1. Find the latest/best checkpoint automatically
output_dir = "./flan-t5-userstories-finetuned"
checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
latest_checkpoint = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
adapter_path = os.path.join(output_dir, latest_checkpoint)

print(f"Loading adapter from: {adapter_path}")

# 2. Load base model in 4-bit (new correct way – no deprecation warnings)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base",
    quantization_config=quant_config,
    device_map="auto"
)

# 3. Load your trained LoRA adapter
model = PeftModel.from_pretrained(model, adapter_path)

# 4. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# 5. Inference function – ONLY OUTPUT
def generate(requirement: str):
    prompt = f"""Convert the following user requirement into detailed User Stories and a Module Breakdown.

User Requirement: {requirement.strip()}"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=512,
        temperature=0.7,
        do_sample=True,
        top_p=0.92,
        repetition_penalty=1.1
    )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# TEST IT NOW – JUST RUN:
generate("As a parent, I want to receive push notifications when my child's school bus is 5 minutes away so that I can be ready at the stop.")

Loading adapter from: ./flan-t5-userstories-finetuned/checkpoint-672
User Stories: - As a parent, I want push notifications so I can check bus times. - As a parent, I want to wait 5 minutes for my bus. - As a parent, I want to wait for the bus to come. - As a parent, I want to monitor traffic. - As an admin, I want to add push notifications so the app is updated. Module Breakdown: - Login/Signup - Bus Tracker - Push Notification System - Bus Notification System - Admin Push Notification System - Admin Dashboard


In [53]:
generate("""
As a freelancer, I want to create and send professional invoices directly from the app with automatic tax calculation and payment reminders so that I get paid faster and look more professional.
""")

User Stories: - As a freelancer, I want to create invoices to track my expenses so I can send invoices. - As a freelancer, I want automatic tax calculations so I can calculate tax on my invoices. - As a freelancer, I want reminders to make payments faster so I can pay faster. - As an admin, I want to add invoice templates so the app is updated. Module Breakdown: - Login/Signup - Invoice Template - Automatic Tax Calculation Module - Payment Reminder System - Admin Template Management


In [54]:
generate("Take photo of houseplant and get instant care tips, watering reminders, and disease detection.")

User Stories: 1. As a user, I want to snap photos of plants to record care tips. 2. As a user, I want watering reminders to check for disease. 3. As a user, I want to see diseases to detect them. 4. As a user, I want to read a care guide to save time. 5. As a user, I want to manage watering patterns. Module Breakdown: Authentication, Registration, Plant Photos, Care Tips, Watering Reminder System, Disease Alert System


In [55]:
generate("Scan any product barcode in store and instantly see if it’s cheaper online with price comparison and reviews.")

User Stories: 1. As a customer, I want to scan products to find price. 2. As a customer, I want price comparison to find cheaper items. 3. As a customer, I want to rate product online. 4. As a customer, I want to compare prices to find cheaper products. 5. As a customer, I want to view reviews to find cheaper items. Module Breakdown: Authentication, Registration, Barcode scanning, Price Comparison, Reviews, Comparison Engine, Rating System


In [ ]:
generate("Scan any product barcode in store and instantly see if it’s cheaper online with price comparison and reviews.")

In [56]:
generate("Create group expense splitting where one person pays and the app automatically calculates who owes what and sends payment requests.")

User Stories: 1. As a user, I want to split expenses so I can pay. 2. As a user, I want to calculate how much I owe. 3. As a user, I want to send payment requests. 4. As a user, I want to check balances. 5. As a user, I want to log expense history. Module Breakdown: Authentication, Registration, Expense Tracker, Expense Log, Balance Tracker, Balance Check, Notification System, Profile Management


In [57]:
generate("I want to track my habits every day.")

User Stories: - As a customer, I want to track my habits so I can stay healthy. - As a customer, I want to see trends so I can know how to stay healthy. - As a customer, I want to see health tips so I can stay healthy. - As an admin, I want to add habits so the app stays updated. Module Breakdown: - Login/Signup - User Authentication - Habit Tracker - Health Tips Newsletter - Health Tips Blog - Admin Dashboard - Notification System


In [58]:
generate("I want an app that helps users track their daily habits, like exercising or drinking wate")

User Stories: - As a user, I want to track my habits so I can improve. - As a user, I want to log workouts so I can track. - As a user, I want to download workouts so I can download. - As an admin, I want to add habits so the app evolves. Module Breakdown: - User Login - Exercise Tracker - Drinking Mode - Admin Habit Management


In [59]:
generate("“I want a website for my local bakery where customers can see all our cakes and breads online.")

User Stories: - As a customer, I want to find cakes and breads so I can order. - As a visitor, I want to view cakes and breads so I can purchase. - As a customer, I want to browse cakes and breads so I can find them. - As an admin, I want to add cakes and breads so the site is updated. Module Breakdown: - Login/Signup - Cake Catalog - Bread Catalog - Admin Dashboard - Contact Us Page


In [61]:
generate("“I want a website in which i see my future work.")

User Stories: - As a user, I want to see my future work. - As a user, I want to share work. - As a user, I want to view projects. - As an admin, I want to manage projects. Module Breakdown: - Login/Signup - Work Gallery - Admin Dashboard - Project Management


In [67]:
generate("Build a website for a sustainable pet food brand to sell eco-friendly products, offer subscriptions, share pet nutrition blogs, and provide a loyalty program.")

User Stories: - As a customer, I want to browse pet food by brand to find eco-friendly options. - As a customer, I want to subscribe for monthly deliveries to save money. - As a customer, I want to read pet nutrition blogs to learn about nutrition. - As a customer, I want a loyalty program to earn rewards. - As an admin, I want to manage subscriptions and inventory to keep the site updated. Module Breakdown: - Login/Signup - Pet Food Catalog - Subscription System - Pet Nutrition Blog - Loyalty Program Module - Admin Dashboard - Contact Us Page


In [64]:
generate("“I want a website where I can browse and buy clothes online.")

User Stories: - As a customer, I want to browse clothes online so I can buy clothes. - As a customer, I want to donate clothes so I can donate. - As a customer, I want to search clothes so I can find clothes. - As an admin, I want to add clothing so the site is updated. Module Breakdown: - Login/Signup - Clothing Catalog - Shopping Site - Payment System - Admin Clothing Management


In [65]:
generate("“We need a fully functional e-commerce website where users can browse, search, and filter a wide range of clothing products by category, size, color, brand, and price. Users should be able to create accounts, log in, and securely manage their profiles, including password recovery and social media login options. The website must allow users to add products to a shopping cart, modify quantities, save items for later, and complete a secure checkout using multiple payment methods. It should provide order tracking, notifications for shipping and delivery, and the ability to leave reviews and ratings for purchased products. The admin panel must enable management of products, categories, inventory, orders, and promotional offers. The website must be mobile-responsive, user-friendly, and ensure the security of user data and payment transactions")

User Stories: - As a customer, I want to browse clothing by category, size, color, brand, and price. - As a user, I want to select a size and color. - As a user, I want to edit quantity. - As a user, I want to save items for later. - As a user, I want to receive notifications for shipping and delivery. - As an admin, I want to manage inventory and orders. Module Breakdown: - User Login - Shopping Cart - Modification Tool - Order Tracking System - Admin Panel Management - Product Catalog - Notification System - Admin Dashboard - Contact Us Page


In [ ]:
We need a simple website for our local bakery to showcase cakes, menus, and allow online inquiries for custom orders.

In [ ]:
generate("“We need a fully functional erce website where users can browse, search, and filter a wide range of clothing products by category, size, color, brand, and price. Users should be able to create accounts, log i")

In [66]:
generate("We need a simple website for our local bakery to showcase cakes, menus, and allow online inquiries for custom orders.")

User Stories: - As a customer, I want to browse cakes to find inspiration. - As a user, I want to read menus to choose favorites. - As a visitor, I want to book online orders to order cakes. - As an admin, I want to manage orders to keep the site updated. Module Breakdown: - Login/Signup - Cake Catalog - Menu Section - Online Order Form - Admin Management Dashboard - Contact Us Page


In [68]:
generate("The client wants a website for an aircraft company that showcases their fleet, services, and aviation solutions. The website should allow visitors to browse different aircraft models with detailed specifications, images, and performance data. Users should be able to request quotes or schedule test flights through an online form. The site should include a news and updates section for aviation industry insights and company announcements. There should be a customer portal for managing bookings, viewing past flight history, and receiving notifications. Admins should be able to manage aircraft details, update news articles, handle customer inquiries, and generate reports on requests and bookings. The website must be responsive, fast-loading, secure, and optimized for SEO. Optional features include a virtual aircraft tour, AI-based aircraft recommendation for clients, and integration with third-party flight management or booking systems.")

User Stories: - As a customer, I want to browse aircraft models to find the right aircraft for my needs. - As a visitor, I want to request a quote or schedule a test flight. - As a user, I want to access news and updates to stay up-to-date. - As an admin, I want to manage aircraft details to ensure a smooth site. - As a customer, I want to access bookings and bookings to manage my bookings. - As an admin, I want to manage aircraft details to keep the site updated. Module Breakdown: - Login/Signup - Aircraft Catalog - Aircraft Catalog - Rental Booking System - Booking Admin Panel - Contact Us Page


In [62]:
# Save the complete model with LoRA weights
output_dir = "./flan-t5-userstories-finetuned"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# Also save the LoRA adapter separately
lora_adapter_path = "./lora-adapter"
model.save_pretrained(lora_adapter_path)

print(f"Model saved to: {output_dir}")
print(f"LoRA adapter saved to: {lora_adapter_path}")

# Create a zip file for easy download
!zip -r trained_model.zip {output_dir} {lora_adapter_path}

Model saved to: ./flan-t5-userstories-finetuned
LoRA adapter saved to: ./lora-adapter
  adding: flan-t5-userstories-finetuned/ (stored 0%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/ (stored 0%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/rng_state.pth (deflated 26%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/training_args.bin (deflated 53%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/optimizer.pt (deflated 8%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/spiece.model (deflated 48%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/scheduler.pt (deflated 62%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/trainer_state.json (deflated 79%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/special_tokens_map.json (deflated 85%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/tokenizer.json (deflated 74%)
  adding: flan-t5-userstories-finetuned/checkpoint-672/adapter_model.safetensors (deflated 7%)
  adding: 

In [63]:
# Download the zip file to your local machine
from google.colab import files
files.download('trained_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>